# Kalman V3.2 — vectorbt Parity FINAL

실제 실패 로그를 기준으로 다시 만든 최종 parity 검증본입니다.

- vectorbt 1.1.0 요구사항에 맞춰 pandas 3.0.5 사용
- 모든 vectorbt 입력을 owned/writable NumPy array로 전달
- `size_type='percent'` 사용
- repo-native CLI를 `python -m`으로 실행
- 실제 vectorbt mini smoke test를 통과한 경우에만 US/KR/BTC 검증 실행
- 실패 시 traceback + child stdout + full log를 Drive에 저장
- V3.2 portfolio 결과는 재계산하지 않음

**Research only / Toss OFF / Neon write OFF / LIVE OFF**


In [ ]:
from google.colab import drive
import json
import shutil
import subprocess
import sys
import traceback
from datetime import datetime
from pathlib import Path

PINNED_SHA = "0946a9d009860fd2ffe4fa948409a28890d64a0a"
SOURCE_BRANCH = "feature/historical-v3-2-portfolio-validation-20260913"
V1_RUN_TAG = "20260913_042850"
V3_CANDIDATE_TAG = "20260913_return_regime_v3_001"
PATCH_TAG = "20260913_v3_2_vectorbt_parity_final_001"

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive')


def run(cmd, *, cwd=None, log_path=None):
    args = [str(x) for x in cmd]
    header = "\n$ " + " ".join(args) + "\n"
    print(header, end="")
    chunks = [header]
    proc = subprocess.Popen(
        args,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        chunks.append(line)
        print(line, end="")
    rc = proc.wait()
    output = "".join(chunks)
    if log_path is not None:
        with Path(log_path).open("a", encoding="utf-8") as fh:
            fh.write(output)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, args, output=output)
    return output


def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=str) + "\n",
        encoding="utf-8",
    )
    tmp.replace(path)


def locate_v1(root, tag):
    for p in [
        root / "Market_Model_V2" / "historical_quant_2017_v1" / tag,
        root / "Kalman" / "Market_Model_V2" / "historical_quant_2017_v1" / tag,
        root / "kalman" / "Market_Model_V2" / "historical_quant_2017_v1" / tag,
    ]:
        if (p / "historical_resume_complete.json").exists():
            return p
    raise FileNotFoundError(f"V1 run not found: {tag}")


def locate_v3(root, tag):
    for p in [
        root / "Market_Model_V2" / "historical_quant_2017_v3_candidate" / tag,
        root / "Kalman" / "Market_Model_V2" / "historical_quant_2017_v3_candidate" / tag,
        root / "kalman" / "Market_Model_V2" / "historical_quant_2017_v3_candidate" / tag,
    ]:
        q = p / "historical_v3_candidate_summary.json"
        if q.exists():
            payload = json.loads(q.read_text(encoding="utf-8"))
            if payload.get("status") == "COMPLETE":
                return p
    raise FileNotFoundError(f"COMPLETE V3 run not found: {tag}")


repo = Path("/content/Codex")
if repo.exists():
    shutil.rmtree(repo)

v1 = locate_v1(DRIVE_ROOT, V1_RUN_TAG)
matrix_dir = v1.parents[1] / "historical_matrices_v1"
v3_root = locate_v3(DRIVE_ROOT, V3_CANDIDATE_TAG)
out_root = (
    v1.parents[1]
    / "historical_quant_2017_v3_2_vectorbt_patch"
    / PATCH_TAG
)
out_root.mkdir(parents=True, exist_ok=True)

full_log = out_root / "parity_full_log.txt"
status_json = out_root / "parity_status.json"
failure_json = out_root / "parity_failure.json"
full_log.write_text("", encoding="utf-8")


def status(state, phase, **extra):
    write_json(status_json, {
        "status": state,
        "phase": phase,
        "updated_at": datetime.now().astimezone().isoformat(),
        "pinned_sha": PINNED_SHA,
        "patch_tag": PATCH_TAG,
        "research_only": True,
        "live_execution": False,
        "toss_execution": False,
        "neon_write": False,
        **extra,
    })


try:
    status("RUNNING", "CLONE")
    run([
        "git", "clone", "--branch", SOURCE_BRANCH,
        "https://github.com/kimtk94/Codex.git", repo,
    ], log_path=full_log)
    run([
        "git", "-C", repo, "checkout", "--detach", PINNED_SHA
    ], log_path=full_log)

    checked = subprocess.check_output(
        ["git", "-C", repo, "rev-parse", "HEAD"],
        text=True,
    ).strip()
    assert checked == PINNED_SHA, (checked, PINNED_SHA)

    app = repo / "kalman-toss-gateway"
    module = app / "research" / "quant_stack" / "historical_v3_2_portfolio_validation.py"
    cli = app / "research" / "quant_stack" / "historical_v3_2_vectorbt_patch.py"
    test_file = app / "tests" / "test_historical_v3_2_portfolio_validation.py"
    spec = app / "config" / "model-v3-historical-return-regime-spec.json"
    for required in (module, cli, test_file, spec):
        assert required.exists(), required

    status("RUNNING", "ISOLATED_ENV")
    if shutil.which("uv") is None:
        run([sys.executable, "-m", "pip", "install", "-q", "uv"], log_path=full_log)
    uv = shutil.which("uv")
    assert uv

    venv = Path("/content/.venv-kalman-v3-2-parity-final")
    if venv.exists():
        shutil.rmtree(venv)
    run([uv, "venv", venv], log_path=full_log)
    vpy = venv / "bin" / "python"

    # vectorbt 1.1.0 explicitly requires pandas >= 3.0.3.
    run([
        uv, "pip", "install", "--python", vpy,
        "pandas==3.0.5",
        "numpy==2.4.6",
        "pyarrow",
        "scipy<1.18",
        "scikit-learn",
        "PyPortfolioOpt==1.6.0",
        "riskfolio-lib==7.3.0",
        "vectorbt==1.1.0",
        "plotly<7",
        "pytest",
    ], log_path=full_log)
    run([uv, "pip", "check", "--python", vpy], log_path=full_log)

    versions = run([
        vpy, "-c",
        (
            "import pandas as pd, numpy as np, vectorbt as vbt; "
            "print('pandas='+pd.__version__); "
            "print('numpy='+np.__version__); "
            "print('vectorbt='+vbt.__version__)"
        ),
    ], cwd=app, log_path=full_log)

    status("RUNNING", "STATIC_AND_SMOKE_TESTS", versions=versions)

    run([vpy, "-m", "py_compile", module, cli, test_file],
        cwd=app, log_path=full_log)

    # Includes the real vectorbt owned-array smoke test.
    run([
        vpy, "-m", "pytest", "-q",
        "tests/test_historical_v3_2_portfolio_validation.py",
    ], cwd=app, log_path=full_log)

    # Extra explicit smoke so the exact vectorbt call is visible in the log.
    smoke = (
        "import numpy as np, vectorbt as vbt; "
        "c=np.array([100.,101.,102.,103.],dtype=float,copy=True); "
        "e=np.array([1,0,0,0],dtype=bool,copy=True); "
        "x=np.array([0,0,1,0],dtype=bool,copy=True); "
        "p=np.array([100.,101.,102.5,103.],dtype=float,copy=True); "
        "pf=vbt.Portfolio.from_signals(c,e,x,price=p,init_cash=1_000_000.,"
        "size=.10,size_type='percent',fees=.0005,slippage=.0005,freq='1D'); "
        "print('SMOKE_TRADES=',int(pf.trades.count())); "
        "print('SMOKE_RETURN=',float(pf.total_return()))"
    )
    run([vpy, "-c", smoke], cwd=app, log_path=full_log)

    status("RUNNING", "FULL_PARITY")

    run([
        vpy, "-m",
        "research.quant_stack.historical_v3_2_vectorbt_patch",
        "--v3-root", v3_root,
        "--matrix-dir", matrix_dir,
        "--spec", spec,
        "--output-dir", out_root,
    ], cwd=app, log_path=full_log)

    summary = out_root / "vectorbt_patch_summary.json"
    assert summary.exists(), summary
    payload = json.loads(summary.read_text(encoding="utf-8"))

    status(
        "COMPLETE",
        "DONE",
        parity_status=payload.get("status"),
        summary=str(summary),
        versions=versions,
    )

    print("\n" + "=" * 88)
    print("KALMAN V3.2 VECTORBT PARITY FINAL COMPLETE")
    print("=" * 88)
    print(summary.read_text(encoding="utf-8"))

except Exception as exc:
    payload = {
        "status": "FAIL",
        "updated_at": datetime.now().astimezone().isoformat(),
        "pinned_sha": PINNED_SHA,
        "error_type": type(exc).__name__,
        "error": str(exc),
        "child_output": getattr(exc, "output", None),
        "traceback": traceback.format_exc(),
        "full_log": str(full_log),
    }
    write_json(failure_json, payload)
    status(
        "FAIL",
        "FAILED",
        error_type=payload["error_type"],
        error=payload["error"],
        failure_json=str(failure_json),
        full_log=str(full_log),
    )
    print("\nFAILURE SAVED:", failure_json)
    print(failure_json.read_text(encoding="utf-8"))
    raise
